In [ ]:
!unzip -q /content/archive.zip -d /content/dataset_children_adults
print("¡Archivo descomprimido con éxito!")

replace /content/dataset_children_adults/test/adults/0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset_children_adults/test/adults/1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: ¡Archivo descomprimido con éxito!


In [ ]:
import os

# Verificar si el archivo existe y su tamaño
if os.path.exists('/content/archive.zip'):
    size_bytes = os.path.getsize('/content/archive.zip')
    size_mb = size_bytes / (1024 * 1024)
    print(f"El archivo existe.")
    print(f"Tamaño del archivo: {size_bytes} bytes ({size_mb:.2f} MB)")
else:
    print("El archivo /content/archive.zip no existe en la ruta especificada.")

El archivo existe.
Tamaño del archivo: 29076475 bytes (27.73 MB)


In [ ]:
# Verificar el tipo de archivo real usando el comando 'file' de Linux
!file /content/archive.zip

/content/archive.zip: Zip archive data, at least v4.5 to extract, compression method=deflate


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import files
import os

# 1. Configurar directorios (apuntando a la carpeta donde ya lo descomprimiste)
DATASET_DIR = '/content/dataset_children_adults'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("Preparando generadores de datos...")
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# 2. Construir el modelo usando Transfer Learning
print("Construyendo el modelo...")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3. Entrenar el modelo
print("Iniciando entrenamiento...")
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

# 4. Guardar y descargar automáticamente el archivo del modelo
nombre_archivo = 'modelo_caras.h5'
model.save(nombre_archivo)
print(f"¡Modelo guardado como {nombre_archivo}!")

files.download(nombre_archivo)

Preparando generadores de datos...
Found 640 images belonging to 2 classes.
Found 160 images belonging to 2 classes.
Construyendo el modelo...
Iniciando entrenamiento...
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 22s 730ms/step - accuracy: 0.8313 - loss: 0.5318 - val_accuracy: 0.8500 - val_loss: 0.4403
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 580ms/step - accuracy: 0.8500 - loss: 0.4280 - val_accuracy: 0.8500 - val_loss: 0.4439
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 12s 585ms/step - accuracy: 0.8500 - loss: 0.4126 - val_accuracy: 0.8500 - val_loss: 0.4361
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 12s 583ms/step - accuracy: 0.8484 - loss: 0.3944 - val_accuracy: 0.8500 - val_loss: 0.4598
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 681ms/step - accuracy: 0.8516 - loss: 0.3869 - val_accuracy: 0.8500 - val_loss: 0.4427


¡Modelo guardado como modelo_caras.h5!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflow as tf

# Cargar el modelo guardado previamente
model = tf.keras.models.load_model('modelo_caras.h5')

# Verificar que la estructura del modelo se cargó correctamente
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,083 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 2 (12.00 B)

In [ ]:
import numpy as np
import os
from tensorflow.keras.preprocessing import image

# Rutas de 5 imágenes de prueba reales (por ejemplo, 3 de children y 2 de adults, o las que prefieras)
# Asegúrate de ajustar 'train/children' o 'train/adults' según la estructura exacta de tus carpetas
imagenes_prueba = [
    '/content/dataset_children_adults/train/children/60.jpg',
    '/content/dataset_children_adults/train/children/61.jpg',
    '/content/dataset_children_adults/train/children/62.jpg',
    '/content/dataset_children_adults/train/adults/60.jpg', # Cambia el número si en adults empieza en otro
    '/content/dataset_children_adults/train/adults/61.jpg'
]

print("--- EVALUANDO 5 CARAS PARA EL REPORTE ---")
for i, img_path in enumerate(imagenes_prueba):
    if os.path.exists(img_path):
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = x / 255.0

        prediccion = model.predict(x)[0][0]
        # Si 0 es children y 1 es adults (o viceversa según el orden alfabético de tus carpetas)
        print(f"Imagen {i+1} ({os.path.basename(os.path.dirname(img_path))}/{os.path.basename(img_path)}): Predicción = {prediccion:.4f}")
    else:
        print(f"No se encontró la imagen en: {img_path}")

--- EVALUANDO 5 CARAS PARA EL REPORTE ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Imagen 1 (children/60.jpg): Predicción = 0.8516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Imagen 2 (children/61.jpg): Predicción = 0.9083
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Imagen 3 (children/62.jpg): Predicción = 0.9110
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Imagen 4 (adults/60.jpg): Predicción = 0.9283
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Imagen 5 (adults/61.jpg): Predicción = 0.7740


In [ ]:
import numpy as np
import os
from tensorflow.keras.preprocessing import image

def evaluar_cinco_rostros(rutas_imagenes, modelo):
    print("--- EVALUACIÓN DE 5 ROSTROS PARA EL REPORTE ---")

    for i, img_path in enumerate(rutas_imagenes):
        if os.path.exists(img_path):
            # 1. Cargar y ajustar tamaño
            img = image.load_img(img_path, target_size=(224, 224))

            # 2. Convertir a matriz y normalizar
            x = image.img_to_array(img)
            x = np.expand_dims(x, axis=0)
            x = x / 255.0

            # 3. Realizar la predicción con el modelo
            prediccion = modelo.predict(x)[0][0]

            # Definir la etiqueta según el valor de probabilidad
            clase = "Adulto / Clase 1" if prediccion > 0.5 else "Niño / Clase 0"

            print(f"[{i+1}] Imagen: {os.path.basename(img_path)}")
            print(f"    -> Resultado predicho: {clase} (Confianza: {prediccion:.4f})")
            print("-" * 50)
        else:
            print(f"[{i+1}] ❌ No se encontró la imagen en la ruta: {img_path}")

# --- EJEMPLO DE USO ---
# Lista con las rutas de 5 imágenes reales de tus carpetas (asegúrate de que existan)
mis_5_caras = [
    '/content/dataset_children_adults/train/children/60.jpg',
    '/content/dataset_children_adults/train/children/61.jpg',
    '/content/dataset_children_adults/train/children/62.jpg',
    '/content/dataset_children_adults/train/adults/60.jpg',
    '/content/dataset_children_adults/train/adults/61.jpg'
]

# Llamamos a la función pasándole tus imágenes y tu modelo entrenado
evaluar_cinco_rostros(mis_5_caras, model)

--- EVALUACIÓN DE 5 ROSTROS PARA EL REPORTE ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
[1] Imagen: 60.jpg
    -> Resultado predicho: Adulto / Clase 1 (Confianza: 0.8516)
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
[2] Imagen: 61.jpg
    -> Resultado predicho: Adulto / Clase 1 (Confianza: 0.9083)
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
[3] Imagen: 62.jpg
    -> Resultado predicho: Adulto / Clase 1 (Confianza: 0.9110)
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
[4] Imagen: 60.jpg
    -> Resultado predicho: Adulto / Clase 1 (Confianza: 0.9283)
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
[5] Imagen: 61.jpg
    -> Resultado predicho: Adulto / Clase 1 (Confianza: 0.7740)
--------------------------------------------------


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
from PIL import Image
import io
import cv2

# Función auxiliar de JavaScript para capturar el video de la webcam
def tomar_foto(quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capturar / Analizar';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "user"}});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Esperar a que el usuario haga clic en el botón
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    image = Image.open(io.BytesIO(binary))
    return np.array(image)

# --- BUCLE DE CLASIFICACIÓN CON TU MODELO ---
try:
    print("📸 Haz clic en el botón para activar la cámara y tomar la foto...")
    # Capturar la imagen desde la webcam del navegador
    imagen_np = tomar_foto()

    # Procesar la imagen para que coincida con lo que espera el modelo (224x224)
    imagen_redimensionada = cv2.resize(imagen_np, (224, 224))
    x = np.expand_dims(imagen_redimensionada, axis=0)
    x = x / 255.0  # Normalizar igual que en el entrenamiento

    # Hacer la predicción con tu modelo entrenado ('model')
    prediccion = model.predict(x)[0][0]

    # Definir el resultado según la salida de la neurona
    clase_detectada = "Adulto (Adults)" if prediccion > 0.5 else "Niño (Children)"
    confianza = prediccion if prediccion > 0.5 else (1 - prediccion)

    print("\n" + "="*40)
    print(f"🎯 RESULTADO DE LA CLASIFICACIÓN EN VIVO:")
    print(f"   -> Categoría: {clase_detectada}")
    print(f"   -> Confianza: {confianza * 100:.2f}%")
    print("="*40)

except Exception as err:
    print(f"Hubo un error al acceder a la cámara o procesar la imagen: {err}")

📸 Haz clic en el botón para activar la cámara y tomar la foto...


<IPython.core.display.Javascript object>

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

🎯 RESULTADO DE LA CLASIFICACIÓN EN VIVO:
   -> Categoría: Adulto (Adults)
   -> Confianza: 89.95%
